# Markowitz Risk-Aversion Allocation

Find a fully invested portfolio that maximizes the sample mean daily return minus a multiple of daily volatility. The example uses dividend- and split-adjusted closing prices from Yahoo Finance and fits its weights on the common available history. The reported return and volatility describe that fitted sample. The weight constraint permits short positions.


In [ ]:
import numpy as np
from scipy.optimize import minimize
import yfinance as yf

symbols = ["MSFT", "COST", "PEP", "RACE"]
risk_aversion = 1.5

history = yf.download(symbols, period="max", auto_adjust=True, progress=False)
if history is None or history.empty:
    raise RuntimeError("Yahoo Finance did not return historical prices.")
close = history["Close"].reindex(columns=symbols).dropna()
if len(close) < 2:
    raise RuntimeError("The selected symbols have insufficient common history.")

returns = close.pct_change().dropna()
expected_returns = returns.mean()
covariance = returns.cov()


def objective(weights):
    daily_return = weights @ expected_returns
    daily_volatility = np.sqrt(weights @ covariance @ weights)
    return -daily_return + risk_aversion * daily_volatility


result = minimize(
    objective,
    np.full(len(symbols), 1 / len(symbols)),
    method="SLSQP",
    constraints={"type": "eq", "fun": lambda weights: weights.sum() - 1},
)
if not result.success:
    raise RuntimeError(result.message)

weights = result.x
print(f"Common price sessions: {len(close):,}")
print("Portfolio weights")
for symbol, weight in zip(symbols, weights):
    print(f"{symbol}: {weight:.2%}")

print(f"Sample mean daily return: {weights @ expected_returns:.4%}")
print(f"Sample daily volatility: {np.sqrt(weights @ covariance @ weights):.4%}")
